In [ ]:
"""
Program Name: loc_calc.ipynb
Description: Uses the inverse square law of sound to determine possible locations for a sound source based on the difference in decibels
Programmer(s): Ben Weinzirl
Date Made: 4/25/2025
Date(s) Revised:
Preconditions: 
Postconditions: 
Errors/Exceptions:
Side Effects:
Invariants: 
Known Faults:
"""

import numpy as np

# ---------- Helper Functions ----------
def distance_ratio_from_db(dB_A, dB_B):
    """Convert dB difference to distance ratio (rB/rA)."""
    return 10 ** ((dB_A - dB_B) / 20)

def find_circle_intersections(c1, r1, c2, r2):
    """Find intersection points of two circles centered at c1, c2 with radii r1, r2."""
    d = np.linalg.norm(c2 - c1)
    if d > r1 + r2 or d < abs(r1 - r2) or d == 0:
        return []
    a = (r1 ** 2 - r2 ** 2 + d ** 2) / (2 * d)
    h = np.sqrt(max(0, r1 ** 2 - a ** 2))
    p2 = c1 + a * (c2 - c1) / d
    offset = h * np.array([-(c2[1] - c1[1]) / d, (c2[0] - c1[0]) / d])
    return [p2 + offset, p2 - offset]

def travel_time(source, mic, speed=5):
    """Calculate time for MPR to travel to source"""
    distance = np.linalg.norm(np.array(source) - np.array(mic))
    return distance / speed


def estimate_direction_and_time(mic_A, mic_B, dB_A, dB_B, speed=5):
    """Estimate sound source positions, direction, and travel times with auto-scaled radii."""
    ratio = distance_ratio_from_db(dB_A, dB_B)
    mic_distance = np.linalg.norm(mic_B - mic_A)

    # 🔁 Auto-scale rA and rB to ensure circles intersect
    rA = mic_distance / (1 + ratio)
    rB = rA * ratio

    intersections = find_circle_intersections(np.array(mic_A), rA, np.array(mic_B), rB)

    if not intersections:
        print("❌ No intersection found. Circles do not intersect.")
        print(f"Mic A at {mic_A} with radius {rA:.4f}")
        print(f"Mic B at {mic_B} with radius {rB:.4f}")
        print(f"Distance between mics: {mic_distance:.4f}")
        return []

    midpoint = (np.array(mic_A) + np.array(mic_B)) / 2
    results = []

    for pt in intersections:
        vec = pt - midpoint
        angle_deg = np.degrees(np.arctan2(vec[1], vec[0]))
        time_to_A = travel_time(pt, mic_A, speed)
        time_to_B = travel_time(pt, mic_B, speed)
        results.append({
            "source_position": pt,
            "angle_deg": angle_deg,
            "time_to_A": time_to_A,
            "time_to_B": time_to_B,
            "time_diff_AB": time_to_A - time_to_B
        })

    return results


# ---------- Main Setup ----------

# Microphone positions (in meters)
# The arrays show the distance between the two locations in meters!!!!
mic_A = np.array([0, 0])
mic_B = np.array([10, 0])

# Decibel readings
# Change the decibels to readings from the MPR recordings!!!
dB_A = 80
dB_B = 74

# Get results
results = estimate_direction_and_time(mic_A, mic_B, dB_A, dB_B)

# ---------- Print Results ----------

print("\n📡 Estimated Sound Source Information:\n")

if results:
    for i, res in enumerate(results):
        source_pos = res["source_position"]
        angle = res["angle_deg"]
        time_to_A = res["time_to_A"]
        time_to_B = res["time_to_B"]
        time_diff_AB = res["time_diff_AB"]

        print(f"--- Possible Sound Source {i + 1} ---")
        print(f"📍 Position:         [{source_pos[0]:.4f}, {source_pos[1]:.4f}]")
        print(f"🧭 Direction Angle:  {angle:.2f}° from midpoint")
        print(f"⏱️  Time from Mic A:    {time_to_A:.6f} s")
        print(f"⏱️  Time from Mic B:    {time_to_B:.6f} s")
        print(f"🔁 Time A - B:        {time_diff_AB:.6f} s")
        print("-" * 40)
else:
    print("No valid sound source locations found due to geometry or input values.")